# Lab 04 — Silver MERGE and idempotent upserts

This notebook promotes the quality-approved candidate batch from `lab04_03_silver_quality.ipynb` into the final Silver Delta table.

## Objectives

- Load only the persisted, validated Silver candidate for the selected batch.
- Enforce the expected Silver column names and data types before writing.
- Create the final Silver table with an explicit schema.
- Classify source rows as new, changed, or unchanged.
- Use Delta Lake `MERGE` to insert new rows and update changed rows.
- Prove that replaying the same batch is idempotent.
- Preserve quality, source, and processing metadata for traceability.

> This is a transactional Silver upsert. Product history and Slowly Changing Dimensions are handled in the following notebooks.

## 1. Load the shared configuration

The configuration notebook supplies the catalog, schema, volume paths, batch ID, contract version, and target table names. Keep the same `batch_id` and `contract_version` used in notebook 03.

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
import sys

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType, DecimalType, IntegerType, LongType,
    StringType, StructField, StructType, TimestampType,
)

lab04_root = (
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/labs/lab_04_silver_quality"
)

if lab04_root not in sys.path:
    sys.path.append(lab04_root)

from src.merge_utils import (
    classify_merge_actions,
    latest_merge_metrics,
    merge_upsert,
)

silver_candidate_path = (
    f"{paths['landing']}/silver_candidates/batch_id={batch_id}"
)

print(f"Candidate path: {silver_candidate_path}")
print(f"Target Silver table: {silver_table}")
print(f"Selected batch: {batch_id}")
print(f"Contract version: {contract_version}")
print("Reusable MERGE module: src.merge_utils")


Candidate path: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/silver_candidates/batch_id=initial
Target Silver table: dbr_dev.parvinbadalov.lab04_silver_transactions
Selected batch: initial
Contract version: v1
Reusable MERGE module: src.merge_utils


## 2. Load the quality-approved candidate batch

Notebook 03 wrote one batch-specific Delta dataset under `landing/silver_candidates`. This boundary prevents invalid Bronze rows from bypassing the quality stage. A missing path means notebook 03 has not completed successfully for the selected batch.

In [0]:
try:
    candidate_df = spark.read.format("delta").load(silver_candidate_path)
except Exception as exc:
    raise FileNotFoundError(
        f"No Silver candidate exists at {silver_candidate_path}. "
        "Run lab04_03_silver_quality.ipynb successfully with the same batch_id first."
    ) from exc

candidate_count = candidate_df.count()
if candidate_count == 0:
    raise ValueError(f"The Silver candidate for batch {batch_id!r} is empty.")

candidate_batch_values = [row[0] for row in candidate_df.select("source_batch_id").distinct().collect()]
if candidate_batch_values != [batch_id]:
    raise AssertionError(
        f"Candidate batch mismatch: expected only {batch_id!r}, found {candidate_batch_values}."
    )

print(f"Candidate rows loaded: {candidate_count:,}")
display(candidate_df.limit(20))

Candidate rows loaded: 315,101


transaction_line_id,invoice_no,stock_code,description,quantity,invoice_timestamp,unit_price,customer_id,country,source_record_hash,source_batch_id,source_file,source_sheet,source_row_number,input_file_path,bronze_ingested_at,quality_contract_version,quality_checked_at,sales_amount,invoice_date,invoice_year,invoice_month,silver_prepared_at
6c4de07354ac2a12840cb7ab18e29b9d0299b88486a9bd6288b25d7347f167f2,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.2500,17850,United Kingdom,d437e01828afac453bca294ee923cdc7ae1985b362293d1004075e5b0cc14b40,initial,Online Retail.xlsx,Online Retail,8,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00004-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-313-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,25.5000,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
7d07613aeedc44e44ab851ad437283d55529b57cf01ca858f2db461bb85de370,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.6500,17850,United Kingdom,0d605467e0138f6396f9c149ce5ad5ab7096c7056518578056ef5dc2dc38dfbd,initial,Online Retail.xlsx,Online Retail,7,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-314-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,15.3000,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
17000e568d97768ddfb50b30790fb358289f6cd675f7925ce9ada92655c8f70b,536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,532942c37207951778ca5258c9b4de0cf4b789f947acdff1ebb7747857250548,initial,Online Retail.xlsx,Online Retail,3,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00006-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-315-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,20.3400,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
f871cfdcba9b6cfe9fa0c049fc80f84ac65f914f8848efe4af499c6fdac859cb,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,5d2f7ca64405415988e613920f8bbd7b3dd24e7632def92f5a560dd284d8d55b,initial,Online Retail.xlsx,Online Retail,6,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00007-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-316-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,20.3400,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
86a800eb6412b8679412adfaebea76891efe154342eaa161ce79ecb41cc1a55a,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,97792215273e2a88a600b1e0c97a2cc746f3dd545256ad9a76ab05e045495e91,initial,Online Retail.xlsx,Online Retail,5,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00008-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-317-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,20.3400,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
6c30ac1498c5debdf9d0ed0ba108ded5e8c27d4ddb9fa285b84e83b0125dd685,536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.8500,17850,United Kingdom,67dbee6035f7cbad53ae0c05b2431427c7772468c33e923cb062a30ad382cfa4,initial,Online Retail.xlsx,Online Retail,9,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00011-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-309-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:18:01.037Z,11.1000,2010-12-01,2010,12,2026-08-10T20:18:01.037Z
8b00013ef5a6440363d5b4c741040095b473b87cd4e29df94bc97d2c193982e4,536367,21754,HOME BUILDING BLOCK WORD,3,2010-12-01T08:34:00.000Z,5.9500,13047,United Kingdom,96adf75ef422460e8562051a0d46b311f0b601cc785640fb26438a04f02d80c6,initial,Online Retail.xlsx,Online Retail,19,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00012-tid-762981857188285749-d78f4527-cbdb-

## 3. Define and validate the Silver data contract

The schema below is intentionally explicit. The notebook checks both missing/extra columns and each column's Spark data type before `MERGE`. This is controlled schema enforcement: an unexpected change fails early instead of silently modifying the analytics table.

Technical merge timestamps are added later and therefore are not part of the candidate contract.

In [0]:
candidate_schema = StructType([
    StructField("transaction_line_id", StringType(), True),
    StructField("invoice_no", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("invoice_timestamp", TimestampType(), True),
    StructField("unit_price", DecimalType(18, 4), True),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("source_record_hash", StringType(), True),
    StructField("source_batch_id", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_sheet", StringType(), True),
    StructField("source_row_number", LongType(), True),
    StructField("input_file_path", StringType(), True),
    StructField("bronze_ingested_at", TimestampType(), True),
    StructField("quality_contract_version", StringType(), True),
    StructField("quality_checked_at", TimestampType(), True),
    StructField("sales_amount", DecimalType(20, 4), True),
    StructField("invoice_date", DateType(), True),
    StructField("invoice_year", IntegerType(), True),
    StructField("invoice_month", IntegerType(), True),
    StructField("silver_prepared_at", TimestampType(), True),
])

expected_types = {field.name: field.dataType.simpleString() for field in candidate_schema.fields}
actual_types = {field.name: field.dataType.simpleString() for field in candidate_df.schema.fields}

missing_columns = sorted(set(expected_types) - set(actual_types))
unexpected_columns = sorted(set(actual_types) - set(expected_types))
type_mismatches = {
    name: {"expected": expected_types[name], "actual": actual_types[name]}
    for name in sorted(set(expected_types) & set(actual_types))
    if expected_types[name] != actual_types[name]
}

if missing_columns or unexpected_columns or type_mismatches:
    raise AssertionError(
        "Silver candidate contract violation. "
        f"Missing={missing_columns}; unexpected={unexpected_columns}; "
        f"type mismatches={type_mismatches}."
    )

candidate_df = candidate_df.select(*[field.name for field in candidate_schema.fields])
print("✅ Silver candidate matches the explicit contract.")
candidate_df.printSchema()

✅ Silver candidate matches the explicit contract.
root
 |-- transaction_line_id: string (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- invoice_timestamp: timestamp (nullable = true)
 |-- unit_price: decimal(18,4) (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_record_hash: string (nullable = true)
 |-- source_batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- source_row_number: long (nullable = true)
 |-- input_file_path: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- quality_contract_version: string (nullable = true)
 |-- quality_checked_at: timestamp (nullable = true)
 |-- sales_amount: decimal(20,4) (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- invoice_

## 4. Validate the business key and required fields

`transaction_line_id` is the stable technical identity inherited from the Bronze source row. `MERGE` requires exactly one source row per key; duplicates would make the update ambiguous. Required business and lineage fields are checked again at the write boundary as defense in depth.

In [0]:
required_candidate_columns = [
    "transaction_line_id", "invoice_no", "stock_code",
    "description", "quantity", "invoice_timestamp",
    "unit_price", "customer_id", "country",
    "source_record_hash", "source_batch_id",
    "quality_contract_version",
]

candidate_quality = candidate_df.agg(
    F.count("*").alias("candidate_rows"),
    F.countDistinct("transaction_line_id").alias("distinct_ids"),
    *[
        F.sum(F.col(name).isNull().cast("long")).alias(f"null_{name}")
        for name in required_candidate_columns
    ],
)
candidate_quality_row = candidate_quality.first().asDict()

if candidate_quality_row["candidate_rows"] != candidate_quality_row["distinct_ids"]:
    raise AssertionError("Candidate transaction_line_id values are not unique.")

null_failures = {
    key: value for key, value in candidate_quality_row.items()
    if key.startswith("null_") and value != 0
}
if null_failures:
    raise AssertionError(f"Required Silver candidate values are null: {null_failures}")

display(candidate_quality)
print("✅ Candidate keys and required fields are valid.")

candidate_rows,distinct_ids,null_transaction_line_id,null_invoice_no,null_stock_code,null_description,null_quantity,null_invoice_timestamp,null_unit_price,null_customer_id,null_country,null_source_record_hash,null_source_batch_id,null_quality_contract_version
315101,315101,0,0,0,0,0,0,0,0,0,0,0,0


✅ Candidate keys and required fields are valid.


## 5. Verify the pre-created Silver target

Permanent table structure is created by **`lab04_00_setup`**, which is run manually and is not part of the production Job.

This Job notebook performs no structural DDL. It validates that the required target exists and that the operational columns needed by the idempotent upsert are present.


In [0]:
if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually when a clean structural rebuild is required."
    )

if not spark.catalog.tableExists(silver_table):
    raise RuntimeError(
        f"Required Silver target does not exist: {silver_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

required_target_columns = {
    *(field.name for field in candidate_schema.fields),
    "silver_created_at",
    "silver_updated_at",
    "silver_last_batch_id",
}

actual_target_columns = set(spark.table(silver_table).columns)
missing_target_columns = sorted(
    required_target_columns - actual_target_columns
)

if missing_target_columns:
    raise AssertionError(
        "Pre-created Silver target is missing columns: "
        + ", ".join(missing_target_columns)
    )

print(f"✅ Pre-created Silver target is ready: {silver_table}")


✅ Pre-created Silver target is ready: dbr_dev.parvinbadalov.lab04_silver_transactions


## 6. Classify the incoming batch

Before changing the table, the notebook calculates expected outcomes:

- **INSERT** — key does not exist in Silver.
- **UPDATE** — key exists but `source_record_hash` changed.
- **UNCHANGED** — key and hash already match.

The null-safe equality operator `<=>` makes hash comparison deterministic even if a malformed historical target contains a null.

In [0]:
merge_classification_df = classify_merge_actions(
    candidate_df,
    silver_table,
    business_keys=["transaction_line_id"],
    hash_column="source_record_hash",
)

merge_plan_df = (
    merge_classification_df
    .groupBy("merge_action")
    .agg(F.count("*").alias("rows"))
    .orderBy("merge_action")
)

merge_plan = {
    row["merge_action"]: row["rows"]
    for row in merge_plan_df.collect()
}

expected_inserts = merge_plan.get("INSERT", 0)
expected_updates = merge_plan.get("UPDATE", 0)
expected_unchanged = merge_plan.get("UNCHANGED", 0)

display(merge_plan_df)

print(
    f"Planned through classify_merge_actions(): "
    f"inserts={expected_inserts:,}, "
    f"updates={expected_updates:,}, "
    f"unchanged={expected_unchanged:,}"
)


merge_action,rows
UNCHANGED,315101


Planned through classify_merge_actions(): inserts=0, updates=0, unchanged=315,101


## 7. Execute the Delta MERGE

Matched rows are updated only when the source hash differs. This condition is essential for idempotency: an unchanged replay does not rewrite every matched row. New keys are inserted with both creation and update timestamps.

In [0]:
business_and_lineage_columns = [
    field.name for field in candidate_schema.fields
]

target_count_before = spark.table(silver_table).count()

merge_upsert(
    candidate_df,
    silver_table,
    business_keys=["transaction_line_id"],
    update_columns=business_and_lineage_columns,
    insert_columns=business_and_lineage_columns,
    hash_column="source_record_hash",
    update_expressions={
        "silver_updated_at": "current_timestamp()",
        "silver_last_batch_id": "source.source_batch_id",
    },
    insert_expressions={
        "silver_created_at": "current_timestamp()",
        "silver_updated_at": "current_timestamp()",
        "silver_last_batch_id": "source.source_batch_id",
    },
)

target_count_after = spark.table(silver_table).count()
actual_insert_growth = target_count_after - target_count_before

if actual_insert_growth != expected_inserts:
    raise AssertionError(
        f"MERGE count mismatch: expected {expected_inserts} new rows, "
        f"but target grew by {actual_insert_growth}."
    )

merge_metrics = latest_merge_metrics(silver_table)

print(f"Rows before MERGE: {target_count_before:,}")
print(f"Rows inserted: {actual_insert_growth:,}")
print(f"Rows expected to update: {expected_updates:,}")
print(f"Rows unchanged: {expected_unchanged:,}")
print(f"Rows after MERGE: {target_count_after:,}")
print(f"Latest Delta operation: {merge_metrics.get('operation')}")
print("✅ Silver upsert executed through src.merge_utils.merge_upsert().")


Rows before MERGE: 315,101
Rows inserted: 0
Rows expected to update: 0
Rows unchanged: 315,101
Rows after MERGE: 315,101
Latest Delta operation: MERGE
✅ Silver upsert executed through src.merge_utils.merge_upsert().


## 8. Validate the merged Silver table

Validation checks global key uniqueness, confirms that every candidate key exists in Silver, and verifies that the stored hash matches the candidate hash. This catches incomplete inserts and incorrect updates.

In [0]:
silver_df = spark.table(silver_table)
silver_count = silver_df.count()
silver_distinct_ids = silver_df.select("transaction_line_id").distinct().count()

if silver_count != silver_distinct_ids:
    raise AssertionError(
        f"Silver key uniqueness failed: rows={silver_count}, distinct IDs={silver_distinct_ids}."
    )

candidate_verification_df = (
    candidate_df.alias("source")
    .join(
        silver_df.select(
            F.col("transaction_line_id").alias("target_transaction_line_id"),
            F.col("source_record_hash").alias("target_source_record_hash"),
        ).alias("target"),
        F.col("source.transaction_line_id") == F.col("target.target_transaction_line_id"),
        "left",
    )
    .agg(
        F.count("*").alias("candidate_rows"),
        F.sum(F.col("target.target_transaction_line_id").isNull().cast("long")).alias("missing_target_rows"),
        F.sum(
            (~F.col("source.source_record_hash").eqNullSafe(F.col("target.target_source_record_hash"))).cast("long")
        ).alias("hash_mismatches"),
    )
)
verification = candidate_verification_df.first().asDict()
if verification["missing_target_rows"] != 0 or verification["hash_mismatches"] != 0:
    raise AssertionError(f"Candidate-to-Silver validation failed: {verification}")

display(candidate_verification_df)
display(
    silver_df
    .filter(F.col("silver_last_batch_id") == batch_id)
    .orderBy("invoice_timestamp", "transaction_line_id")
    .limit(30)
)
print("✅ Silver MERGE validation passed.")

candidate_rows,missing_target_rows,hash_mismatches
315101,0,0


transaction_line_id,invoice_no,stock_code,description,quantity,invoice_timestamp,unit_price,customer_id,country,source_record_hash,source_batch_id,source_file,source_sheet,source_row_number,input_file_path,bronze_ingested_at,quality_contract_version,quality_checked_at,sales_amount,invoice_date,invoice_year,invoice_month,silver_prepared_at,silver_created_at,silver_updated_at,silver_last_batch_id
17000e568d97768ddfb50b30790fb358289f6cd675f7925ce9ada92655c8f70b,536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,532942c37207951778ca5258c9b4de0cf4b789f947acdff1ebb7747857250548,initial,Online Retail.xlsx,Online Retail,3,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00006-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-315-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,20.3400,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08-09T20:55:06.460Z,initial
6c4de07354ac2a12840cb7ab18e29b9d0299b88486a9bd6288b25d7347f167f2,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.2500,17850,United Kingdom,d437e01828afac453bca294ee923cdc7ae1985b362293d1004075e5b0cc14b40,initial,Online Retail.xlsx,Online Retail,8,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00004-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-313-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,25.5000,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08-09T20:55:06.460Z,initial
7d07613aeedc44e44ab851ad437283d55529b57cf01ca858f2db461bb85de370,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.6500,17850,United Kingdom,0d605467e0138f6396f9c149ce5ad5ab7096c7056518578056ef5dc2dc38dfbd,initial,Online Retail.xlsx,Online Retail,7,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-314-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,15.3000,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08-09T20:55:06.460Z,initial
86a800eb6412b8679412adfaebea76891efe154342eaa161ce79ecb41cc1a55a,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,97792215273e2a88a600b1e0c97a2cc746f3dd545256ad9a76ab05e045495e91,initial,Online Retail.xlsx,Online Retail,5,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00008-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-317-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,20.3400,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08-09T20:55:06.460Z,initial
ebb5304c4d44f54ef8cb0c618c67b7c0439fdcce9f5c05922ec85d1d4bdbb90c,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.7500,17850,United Kingdom,c1cd806692dd9c0f64a38acc337200ebc4eff66727b38ab69269e4a98bd4744f,initial,Online Retail.xlsx,Online Retail,4,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00009-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-318-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,22.0000,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08-09T20:55:06.460Z,initial
f871cfdcba9b6cfe9fa0c049fc80f84ac65f914f8848efe4af499c6fdac859cb,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,5d2f7ca64405415988e613920f8bbd7b3dd24e7632def92f5a560dd284d8d55b,initial,Online Retail.xlsx,Online Retail,6,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00007-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-316-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-09T20:47:50.834Z,20.3400,2010-12-01,2010,12,2026-08-09T20:47:50.834Z,2026-08-09T20:55:06.460Z,2026-08

✅ Silver MERGE validation passed.


## 9. Replay the same batch to prove idempotency

The exact same `MERGE` is executed again. Because all source keys and hashes now match, the row count must remain unchanged. This demonstrates safe job repair and scheduled reruns.

In [0]:
count_before_replay = spark.table(silver_table).count()

merge_upsert(
    candidate_df,
    silver_table,
    business_keys=["transaction_line_id"],
    update_columns=business_and_lineage_columns,
    insert_columns=business_and_lineage_columns,
    hash_column="source_record_hash",
    update_expressions={
        "silver_updated_at": "current_timestamp()",
        "silver_last_batch_id": "source.source_batch_id",
    },
    insert_expressions={
        "silver_created_at": "current_timestamp()",
        "silver_updated_at": "current_timestamp()",
        "silver_last_batch_id": "source.source_batch_id",
    },
)

count_after_replay = spark.table(silver_table).count()

if count_after_replay != count_before_replay:
    raise AssertionError(
        f"Silver replay was not idempotent: "
        f"before={count_before_replay}, after={count_after_replay}."
    )

replay_classification_df = classify_merge_actions(
    candidate_df,
    silver_table,
    business_keys=["transaction_line_id"],
    hash_column="source_record_hash",
)

replay_changed_rows = replay_classification_df.filter(
    F.col("merge_action").isin("INSERT", "UPDATE")
).count()

if replay_changed_rows != 0:
    raise AssertionError(
        f"Expected zero pending changes after replay, "
        f"found {replay_changed_rows}."
    )

print(
    "✅ Idempotency passed through merge_upsert(): "
    "replay inserted 0 rows and left 0 changed rows."
)
print(f"Silver row count remains {count_after_replay:,}.")


✅ Idempotency passed through merge_upsert(): replay inserted 0 rows and left 0 changed rows.
Silver row count remains 315,101.


## 10. Inspect Delta history and final results

Delta history provides evidence of table creation and `MERGE` operations. Operation metrics can be expanded to inspect inserted, updated, deleted, and copied rows.

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {silver_table}")
display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics",
    )
    .orderBy(F.col("version").desc())
    .limit(10)
)

final_validation_df = spark.createDataFrame(
    [
        ("candidate_rows", candidate_count),
        ("planned_inserts", expected_inserts),
        ("planned_updates", expected_updates),
        ("planned_unchanged", expected_unchanged),
        ("silver_rows_after_merge", target_count_after),
        ("silver_rows_after_replay", count_after_replay),
        ("replay_changed_rows", replay_changed_rows),
    ],
    ["validation", "result"],
)

display(final_validation_df)
print("✅ Silver upsert and replay tests completed successfully.")


version,timestamp,operation,operationParameters,operationMetrics
8,2026-08-10T20:26:54.000Z,MERGE,"Map(predicate -> [""(transaction_line_id#24981 = transaction_line_id#21919)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#24990 <=> source_record_hash#21928)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4191, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1590, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 315101, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2564)"
7,2026-08-10T20:26:41.000Z,MERGE,"Map(predicate -> [""(transaction_line_id#22945 = transaction_line_id#21919)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#22954 <=> source_record_hash#21928)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4656, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2075, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 315101, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2544)"
6,2026-08-10T02:06:57.000Z,MERGE,"Map(predicate -> [""(transaction_line_id#23672 = transaction_line_id#20944)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#23681 <=> source_record_hash#20953)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4162, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1657, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 315101, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2464)"
5,2026-08-10T02:06:46.000Z,MERGE,"Map(predicate -> [""(transaction_line_id#21824 = transaction_line_id#20944)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#21833 <=> source_record_hash#20953)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5303, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2396, numTargetRowsUpdated -> 0, numOutputRows -> 0, numT

validation,result
candidate_rows,315101
planned_inserts,0
planned_updates,0
planned_unchanged,315101
silver_rows_after_merge,315101
silver_rows_after_replay,315101
replay_changed_rows,0


✅ Silver upsert and replay tests completed successfully.


## What this notebook demonstrated

- The quality-approved batch was isolated from raw Bronze data.
- An explicit contract rejected unexpected Silver candidate schemas.
- Stable technical keys made the Delta `MERGE` deterministic.
- New rows were inserted and changed rows were updated.
- Unchanged rows were not rewritten.
- A replay produced no duplicates and no row-count change.
- Delta history recorded the table operations.

## Next notebook

Continue with **`lab04_05_scd_type1.ipynb`**. It will build a current-state product dimension keyed by `stock_code`, apply SCD Type 1 overwrite behavior with `MERGE`, and validate that only the latest product attributes remain.